In [1]:
# !pip install chromadb
# !pip install -U sentence-transformers transformers huggingface_hub accelerate


In [2]:
from pathlib import Path
import json

import chromadb
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

/home/milijanas/projects/genomic-rag-assistant/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
project_root = Path("/home/milijanas/projects/genomic-rag-assistant")
data_raw = project_root / "data/raw"
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

In [4]:

CLINVAR_PARQUET = data_raw / "clinvar_clean_snv_grch38.parquet"
CLINVAR_EXPORT = data_processed / "docs_for_rag_clinvar.jsonl"
CLINVAR_COLLECTION = "clinvar_chunks"
CLINVAR_EMBEDDINGS = data_processed / "clinvar_chunk_embeddings.npy"

CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"


## Offline — build index

Run rarely: `DOWNLOAD` rebuilds parquet from NCBI; `REBUILD_INDEX` re-embeds Chroma.

Set `CHUNK_MODE` to `"merged"` (1 chunk/variant) or `"by_field"` (identity + one chunk per field).


In [5]:
# OFFLINE: download and filter ClinVar -> parquet
DOWNLOAD = False

if DOWNLOAD:
    columns = [
        "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
        # "ReferenceAllele", "AlternateAllele",
        "ClinicalSignificance", "ReviewStatus",
        "Type", "PhenotypeList", "PhenotypeIDS", "Assembly", "HGNC_ID",
        "NumberSubmitters", "LastEvaluated",
    ]
    dtype_map = {
        "VariationID": "int32", "Name": "string", "GeneSymbol": "category",
        "Chromosome": "category", "Start": "Int64", "Stop": "Int64",
        # "ReferenceAllele": "string", "AlternateAllele": "string",
        "ClinicalSignificance": "category", "ReviewStatus": "category",
        "Type": "category", "PhenotypeList": "string", "PhenotypeIDS": "string",
        "Assembly": "category", "HGNC_ID": "string", "NumberSubmitters": "Int16",
    }
    allowed_review = {
        "practice guideline",
        "reviewed by expert panel",
        "criteria provided, multiple submitters, no conflicts",
    }
    chunks = []
    for chunk in pd.read_csv(
        CLINVAR_URL, sep="\t", compression="gzip", usecols=columns,
        dtype=dtype_map, chunksize=100_000, low_memory=True,
    ):
        filtered = chunk[
            # (chunk["Type"] == "single nucleotide variant")
              (chunk["Assembly"] == "GRCh38")
            & (chunk["ReviewStatus"].isin(allowed_review))
        ]
        chunks.append(filtered)
    df_build = pd.concat(chunks, ignore_index=True).drop_duplicates()
    data_raw.mkdir(parents=True, exist_ok=True)
    df_build.to_parquet(CLINVAR_PARQUET, index=False)
    print("Saved:", CLINVAR_PARQUET, df_build.shape)
else:
    print("Skipping download:", CLINVAR_PARQUET)

Skipping download: /home/milijanas/projects/genomic-rag-assistant/data/raw/clinvar_clean_snv_grch38.parquet


In [6]:
# OFFLINE: load parquet and subset for indexing
df = pd.read_parquet(CLINVAR_PARQUET)

In [7]:
def parse_phenotypes(x):
    if pd.isna(x):
        return 'not provided'

    r = x.replace("not provided|","").replace("not specified|","").replace("|not provided","").replace("|not specified","")
    r = r.replace('not specified','not provided')
    r = r.replace("|",". ")
    
    return r

df['PhenotypeList'] = df['PhenotypeList'].apply(parse_phenotypes)

In [8]:
df.head(1)

,Type,Name,GeneSymbol,HGNC_ID,ClinicalSignificance,LastEvaluated,PhenotypeIDS,PhenotypeList,Assembly,Chromosome,Start,Stop,ReviewStatus,NumberSubmitters,VariationID
0,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,"Dec 17, 2024","MONDO:MONDO:0013342,MedGen:C3150901,OMIM:61364...",Hereditary spastic paraplegia 48. Macular dyst...,GRCh38,7,4781213,4781216,"criteria provided, multiple submitters, no con...",4,2


In [9]:
df['Type'].value_counts(dropna=False).head(20)

Type
single nucleotide variant    633352
Deletion                      26068
Duplication                   11507
Microsatellite                 7596
Indel                          2512
Insertion                      1575
Inversion                       207
Variation                        73
copy number loss                 25
copy number gain                 11
Name: count, dtype: int64

In [10]:
# df['ReferenceAllele'].value_counts(dropna=False).head(20) # all empty --> drop column
# df['AlternateAllele'].value_counts(dropna=False).head(20) # all empty --> drop column


In [11]:
df['ReviewStatus'].value_counts(dropna=False) # practice guideline should be bigger? is it due to snp filter?

ReviewStatus
criteria provided, multiple submitters, no conflicts    661056
reviewed by expert panel                                 21816
practice guideline                                          54
Name: count, dtype: int64

In [12]:

df['ClinicalSignificance'].value_counts(dropna=False)

ClinicalSignificance
Uncertain significance                                                                          299033
Likely benign                                                                                   130872
Benign                                                                                           93340
Benign/Likely benign                                                                             63895
Pathogenic                                                                                       41854
Pathogenic/Likely pathogenic                                                                     39106
Likely pathogenic                                                                                14364
Uncertain significance/Uncertain risk allele                                                       142
drug response                                                                                      106
Pathogenic; drug response                           

In [13]:

df['PhenotypeList'].value_counts(dropna=False).head(20)

PhenotypeList
not provided                                                                             121376
Inborn genetic diseases                                                                   44100
Hereditary cancer-predisposing syndrome                                                   11797
Cardiovascular phenotype                                                                   5801
Hereditary cancer-predisposing syndrome. Familial adenomatous polyposis 1                  3215
Hereditary cancer-predisposing syndrome. Familial cancer of breast                         2921
Familial cancer of breast. Hereditary cancer-predisposing syndrome                         2555
Primary ciliary dyskinesia                                                                 2447
Hereditary nonpolyposis colorectal neoplasms. Hereditary cancer-predisposing syndrome      2079
Hereditary cancer-predisposing syndrome. Hereditary nonpolyposis colorectal neoplasms      1821
Ataxia-telangiectasia synd

In [ ]:
df_tmp = df[
    df["PhenotypeList"].isna() | (df["PhenotypeList"] == "not provided") | (df["PhenotypeList"] == "not specified")
]
df_tmp['ClinicalSignificance'].value_counts(dropna=False)


ClinicalSignificance
Benign                          52125
Uncertain significance          32975
Likely benign                   25778
Benign/Likely benign             8357
Pathogenic                        969
Pathogenic/Likely pathogenic      849
Likely pathogenic                 317
Likely benign; other                4
Benign; other                       1
Likely benign; association          1
Name: count, dtype: int64

In [ ]:
df_tmp = df[
    df["ClinicalSignificance"] == "Uncertain significance"
]
df_tmp['PhenotypeList'].value_counts(dropna=False)

In [ ]:
del df_tmp

In [15]:

# For RAG - filter to highest review tiers only (~15k variants) and non-VUS
df = df[df["ReviewStatus"].isin(["practice guideline", "reviewed by expert panel"])].copy()
df = df[df["ClinicalSignificance"] != 'Uncertain significance'].copy()
df = df[df["Type"] == 'single nucleotide variant'].copy()

# df = df.head(1000)  # dev subsample

print(df.shape)


(11750, 15)


In [16]:
def _is_valid_field(val) -> bool:
    if pd.isna(val):
        return False
    s = str(val).strip().lower()
    return bool(s) and s != "nan"


def _col_value(row, column: str) -> str:
    return str(row[column]) if _is_valid_field(row.get(column)) else ""


FIELD_SPECS = [
    ("gene", "GeneSymbol"),
    ("pathogenicity", "ClinicalSignificance"),
    ("impact", "PhenotypeList"),
    ("variant type", "Type"),
    ("reliablity", "ReviewStatus"),
]


def _row_field_values(row) -> dict:
    field_values = {chunk_type: _col_value(row, col) for chunk_type, col in FIELD_SPECS}
    phenotypes = parse_phenotypes(row.get("PhenotypeList"))
    field_values["impact"] = "|".join(phenotypes) if phenotypes else ""
    return field_values


def build_chunked_docs(dataframe: pd.DataFrame, chunk_mode: str = "by_field") -> list:
    if chunk_mode not in {"by_field", "merged"}:
        raise ValueError("chunk_mode must be 'by_field' or 'merged'")

    chunked = []
    for _, row in dataframe.iterrows():
        variation_id = str(row["VariationID"])
        field_values = _row_field_values(row)
        gene = field_values["gene"]
        header = f"Variation: {variation_id}"

        if chunk_mode == "by_field":
            chunked.append({
                "id": f"{variation_id}_identity",
                "text": f"{header}\nType: identity",
                "metadata": {"variation_id": variation_id, "chunk_type": "identity", "gene": gene},
            })
            for chunk_type, column in FIELD_SPECS:
                value = field_values[chunk_type]
                if not value:
                    continue
                chunked.append({
                    "id": f"{variation_id}_{chunk_type.replace(' ', '_')}",
                    "text": f"{header}\nType: {chunk_type}\n{value}",
                    "metadata": {
                        "variation_id": variation_id,
                        "chunk_type": chunk_type,
                        "gene": gene,
                        column: value,
                    },
                })
        else:
            summary_lines = [
                f"{chunk_type}: {field_values[chunk_type]}"
                for chunk_type, _ in FIELD_SPECS
                if field_values[chunk_type]
            ]
            if not summary_lines:
                continue
            metadata = {
                "variation_id": variation_id,
                "chunk_type": "summary",
                "gene": gene,
            }
            for chunk_type, column in FIELD_SPECS:
                if field_values[chunk_type]:
                    metadata[column] = field_values[chunk_type]
            chunked.append({
                "id": f"{variation_id}_summary",
                "text": f"{header}\nType: summary\n" + "\n".join(summary_lines),
                "metadata": metadata,
            })
    return chunked


In [ ]:
# OFFLINE: chunk, export, embed, Chroma
CHUNK_MODE = "merged"  # "merged" | "by_field"
REBUILD_INDEX = True
CHROMA_BATCH_SIZE = 5000
EMBED_BATCH_SIZE = 64

docs = build_chunked_docs(df, chunk_mode=CHUNK_MODE)
print(f"Chunk mode: {CHUNK_MODE}")
print(f"Built {len(docs)} chunks from {len(df)} variants (~{len(docs) / len(df):.1f} chunks/variant)")

data_processed.mkdir(parents=True, exist_ok=True)
chroma_dir.mkdir(parents=True, exist_ok=True)

with CLINVAR_EXPORT.open("w", encoding="utf-8") as f:
    for d in docs:
        f.write(json.dumps({"id": d["id"], "text": d["text"], "metadata": d["metadata"]}, ensure_ascii=False) + "\n")

ids = [d["id"] for d in docs]
documents = [d["text"] for d in docs]
metadatas = [d["metadata"] for d in docs]

client = chromadb.PersistentClient(path=str(chroma_dir))

if REBUILD_INDEX:
    model = SentenceTransformer("BAAI/bge-small-en-v1.5")
    embeddings = model.encode(documents, batch_size=EMBED_BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
    np.save(CLINVAR_EMBEDDINGS, embeddings)

    try:
        client.delete_collection(CLINVAR_COLLECTION)
    except Exception:
        pass
    collection = client.create_collection(CLINVAR_COLLECTION)
    for i in range(0, len(ids), CHROMA_BATCH_SIZE):
        collection.add(
            ids=ids[i : i + CHROMA_BATCH_SIZE],
            documents=documents[i : i + CHROMA_BATCH_SIZE],
            embeddings=embeddings[i : i + CHROMA_BATCH_SIZE].tolist(),
            metadatas=metadatas[i : i + CHROMA_BATCH_SIZE],
        )
    print(f"Indexed {len(ids)} chunks -> {CLINVAR_COLLECTION}")
else:
    print("REBUILD_INDEX is False — skip embed/Chroma.")


Chunk mode: merged
Built 11750 chunks from 11750 variants (~1.0 chunks/variant)


Batches:   8%|▊         | 15/184 [04:17<46:06, 16.37s/it]

## Online — query only


In [ ]:
# ONLINE: query
query = "pathogenic variants related to breast cancer"
TOP_K_CHUNKS = 20
TOP_K_VARIANTS = 5

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = chromadb.PersistentClient(path=str(chroma_dir))
collection = client.get_collection(CLINVAR_COLLECTION)
print(f"Collection {CLINVAR_COLLECTION}: {collection.count()} chunks")

results = collection.query(
    query_embeddings=[model.encode(query).tolist()],
    n_results=TOP_K_CHUNKS,
    include=["documents", "metadatas", "distances"],
)

chunks_by_variant = {}
best_by_variant = {}
for doc_text, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0],
):
    vid = (meta or {}).get("variation_id")
    hit = {"text": doc_text, "metadata": meta, "distance": dist}
    chunks_by_variant.setdefault(vid, []).append(hit)
    if vid not in best_by_variant or dist < best_by_variant[vid]["distance"]:
        best_by_variant[vid] = hit

ranked = sorted(best_by_variant.values(), key=lambda x: x["distance"])[:TOP_K_VARIANTS]
print(f"Query: {query}\n")
for rank, hit in enumerate(ranked, 1):
    m = hit["metadata"] or {}
    n_chunks = len(chunks_by_variant.get(m.get("variation_id"), []))
    print(
        f"{rank}. variation_id={m.get('variation_id')} gene={m.get('gene')} "
        f"best_chunk={m.get('chunk_type')} dist={hit['distance']:.4f} "
        f"({n_chunks} retrieved chunk(s))"
    )
    print(hit["text"][:500] + ("..." if len(hit["text"]) > 500 else ""))
    print()


In [ ]:
# ONLINE: combine retrieved chunks with full ClinVar records
if "ranked" not in globals():
    raise ValueError("Run the query cell above first.")

CLINVAR_RECORD_FIELDS = [
    "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
    "Type", "ClinicalSignificance", "ReviewStatus", "PhenotypeList",
    "PhenotypeIDS", "Assembly", "HGNC_ID", "NumberSubmitters", "LastEvaluated",
]


def format_variant_record(row: pd.Series) -> str:
    return "\n".join(
        f"{col}: {row[col]}"
        for col in CLINVAR_RECORD_FIELDS
        if col in row.index and pd.notna(row[col])
    )


def build_rag_context(query: str, ranked: list, chunks_by_variant: dict, records_df: pd.DataFrame) -> str:
    sections = []
    for rank, hit in enumerate(ranked, 1):
        meta = hit.get("metadata") or {}
        vid = str(meta.get("variation_id"))
        row = records_df.loc[records_df["VariationID"] == int(vid)]
        full_record = format_variant_record(row.iloc[0]) if len(row) else "(record not found)"

        variant_chunks = sorted(
            chunks_by_variant.get(vid, [hit]),
            key=lambda x: x["distance"],
        )
        chunk_parts = []
        for i, chunk in enumerate(variant_chunks, 1):
            cm = chunk.get("metadata") or {}
            chunk_parts.append(
                f"Chunk {i} ({cm.get('chunk_type')}, dist={chunk['distance']:.4f}):\n{chunk['text']}"
            )

        sections.append(
            f"### Result {rank} — variation_id={vid}, gene={meta.get('gene')}\n\n"
            f"Retrieved chunks:\n"
            + "\n\n".join(chunk_parts)
            + "\n\nFull ClinVar record:\n"
            + full_record
        )
    return f"User query: {query}\n\n" + "\n\n---\n\n".join(sections)


if "df" not in globals():
    raise ValueError("Run the offline load/filter cells first so `df` is in memory.")

rag_context = build_rag_context(query, ranked, chunks_by_variant, df)
print(rag_context[:4000] + ("..." if len(rag_context) > 4000 else ""))


In [ ]:
# ONLINE: summarize with Llama 3.1 8B Instruct
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
LOAD_LLM = True
MAX_NEW_TOKENS = 1024

SYSTEM_PROMPT = (
    "You are a clinical genomics assistant. Summarize ClinVar variant search results "
    "for the user's query using ONLY the retrieved chunks and full records provided. "
    "Be concise, accurate, and structured. For each relevant variant, mention gene, "
    "variant name, clinical significance, phenotypes, review status, and variation ID. "
    "If information is missing, say so. Do not invent facts."
)


def summarize_rag(query: str, context: str, model, tokenizer, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Query: {query}\n\n"
                f"Retrieved evidence:\n\n{context}\n\n"
                "Write a short summary answering the query from this evidence."
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output_ids[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


if "rag_context" not in globals():
    raise ValueError("Run the context cell above first.")

if LOAD_LLM:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("Warning: no GPU — Llama 3.1 8B on CPU is slower but much lighter than 14B (~16GB RAM).")

    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        model = model.to(device)

    summary = summarize_rag(query, rag_context, model, tokenizer)
    print("=" * 60)
    print(f"Query: {query}\n")
    print(summary)
else:
    print("LOAD_LLM is False — set True to run Llama 3.1 8B summarization.")